# Solar Filament Segmentation — Phase 2 baseline (Mask R-CNN R50-FPN) on Google Colab

Trains the first real model, scores it on the frozen validation split with the **official Kaggle metric**,
tunes the score threshold / minimum area on validation, and writes the first submission CSV.

| Step | Time on a T4 (estimate) |
|---|---|
| Install + Detectron2 build | 5-8 min |
| Training, 6,000 iterations | ~1-1.5 h |
| Validation (141 images) | ~5 min |
| Sweep (141 images, once) | ~5 min |
| Test inference (180 images) | ~5 min |

**Before you start**
1. `Runtime -> Change runtime type -> T4 GPU` (or better).
2. Colab secret `GITHUB_TOKEN` if the repo is private (same as the foundation notebook).
3. Google Drive: checkpoints and results are saved to `MyDrive/solar_filament/`, so a disconnected
   session loses nothing. **If Colab disconnects:** run cells 1-4 again, then the training cell;
   it resumes from the last checkpoint (saved every 1,000 iterations).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Mount Google Drive (persistent storage)

In [ ]:
import os
from google.colab import drive
drive.mount("/content/drive")
DRIVE_DIR = "/content/drive/MyDrive/solar_filament"
OUT_DIR = f"{DRIVE_DIR}/output"
os.makedirs(OUT_DIR, exist_ok=True)
os.environ["SOLAR_OUTPUT_DIR"] = OUT_DIR   # checkpoints, val results, sweep -> Drive
print("Results go to", OUT_DIR)

## 2. Clone the repository

In [ ]:
BRANCH = "Dev"
REPO = "github.com/amiruliman592/Kaggle-Competition-.git"
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None
url = f"https://{token}@{REPO}" if token else f"https://{REPO}"

if not os.path.exists("/content/solar"):
    !git clone -q --depth 1 -b {BRANCH} {url} /content/solar
%cd /content/solar
os.environ["SOLAR_ROOT"] = "/content/solar"
!git log --oneline -1
!ls filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026/train/train_images | wc -l   # expect 707
!ls filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026/test/test_images | wc -l    # expect 180

## 3. Install dependencies (Detectron2 build takes ~5-8 min)

In [ ]:
!pip install -q -r requirements.txt
!pip install -q --no-build-isolation 'git+https://github.com/facebookresearch/detectron2.git'
import torch, detectron2
print("torch", torch.__version__, "| detectron2", detectron2.__version__)
!head -1 requirements-lock.txt   # versions of the Phase 0 dry run, for comparison

## 4. Helper: copy small results to Drive
The experiment log and submissions live inside the repo folder (lost on disconnect), so they are copied
to Drive after each step.

In [ ]:
import shutil, glob
def save_to_drive():
    os.makedirs(f"{DRIVE_DIR}/submissions", exist_ok=True)
    shutil.copy("experiments/log.csv", f"{DRIVE_DIR}/log.csv")
    for f in glob.glob("submissions/*.csv"):
        shutil.copy(f, f"{DRIVE_DIR}/submissions/")
    print("Saved log + submissions to", DRIVE_DIR)

## 5. Test suite (expect `21 passed`)

In [ ]:
!python -m pytest -q -p no:warnings tests

## 6. Train Mask R-CNN R50-FPN (~1-1.5 h)
Settings come from `src/config.py`: anchors 8-128 with ratios 0.2-5, 1024 px input, LR 0.0025, batch 2,
6,000 iterations, flips + brightness/contrast. A warning that some RPN head weights were skipped
(shape mismatch) is **expected**: the 5 aspect ratios need a new RPN head.

Watch `total_loss` in the log: it should fall clearly over the first ~1,000 iterations.
Re-running this cell after a disconnect resumes from the last checkpoint.

In [ ]:
!python run_pipeline.py --step train --name baseline_r50

## 7. Check the checkpoint and loss curve

In [ ]:
import json
import matplotlib.pyplot as plt
ckpt_dir = f"{OUT_DIR}/detectron2/baseline_r50"
!ls -la {ckpt_dir}
rows = [json.loads(l) for l in open(f"{ckpt_dir}/metrics.json") if "total_loss" in l]
plt.plot([r["iteration"] for r in rows], [r["total_loss"] for r in rows])
plt.xlabel("iteration"); plt.ylabel("total_loss"); plt.title("baseline_r50 training loss"); plt.show()

## 8. Validation with the default thresholds (score 0.5, min area 50)
`official_PQ` is the Kaggle metric (all annotators). It is the number logged as `val_pq`.
The plain `PQ` row uses one annotator only and is for diagnostics (fragmentation, merges, size bins).

In [ ]:
!python run_pipeline.py --step val --name baseline_r50
save_to_drive()

## 9. Sweep score threshold x minimum area on validation
Predicts the validation images once, then tries every combination in `CFG.infer.SWEEP_*`.
Prints the top 10 by official PQ and saves `sweep_baseline_r50.json` to Drive.

In [ ]:
!python run_pipeline.py --step sweep --name baseline_r50

## 10. Validate + write the submission with the best sweep setting
Re-runs validation with the chosen setting (so the logged row matches the submission),
then predicts the 180 test images and checks the CSV with `validate_submission.py`.

In [ ]:
best = json.load(open(f"{OUT_DIR}/sweep_baseline_r50.json"))[0]
THR, AREA = best["score_threshold"], best["min_area"]
print(f"Using --score-thr {THR} --min-area {AREA} (val official PQ {best['PQ']:.4f})")
!python run_pipeline.py --step val   --name baseline_r50 --score-thr {THR} --min-area {AREA}
!python run_pipeline.py --step infer --name baseline_r50 --score-thr {THR} --min-area {AREA}
save_to_drive()

## 11. Send results back
1. Download the files below (they are also in `MyDrive/solar_filament/`).
2. Upload `submission_baseline_r50.csv` to the competition: **Submit Prediction** on the Kaggle competition page.
3. Paste into Claude: the training-loss plot (or last loss values), the printed val results of cells 8 and 10,
   the sweep table, and the Kaggle public score.

In [ ]:
from google.colab import files
files.download("experiments/log.csv")
files.download("submissions/submission_baseline_r50.csv")
files.download(f"{OUT_DIR}/sweep_baseline_r50.json")
files.download(f"{OUT_DIR}/val_results_baseline_r50.json")